# Model Selection — Binary Classification of Obesity Risk

### 🔍 Project Context
This project applies an end‑to‑end machine learning workflow to a structured health dataset, with the goal of predicting obesity risk based on behavioral and demographic attributes.
The focus is on building a practical, decision‑oriented model, aligned with applied analytics and Business Intelligence use cases, rather than purely academic classification.

[UCI Repository](https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition)

### 🎯 Problem Definition

The original multi‑class obesity labels are reformulated into a binary classification task:

0 — No Obesity Risk
(Normal Weight + Insufficient Weight)

1 — Obesity Risk
(Overweight + Obesity categories)

This framing supports real‑world screening scenarios, where identifying individuals at risk is more relevant than fine‑grained categorical distinctions.

#### Model Selection Goal

The objective of this notebook is to select the most appropriate classification model using AutoSklearn2 with cross‑validation.

This notebook focuses exclusively on model selection.
Hyperparameter optimization and final model evaluation are performed in subsequent notebooks.

In [1]:
from auto_sklearn2 import AutoSklearnClassifier

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score

#### Load Data

In [2]:
import pandas as pd
from pathlib import Path

# Define project root dynamically
PROJECT_ROOT = Path.cwd()

# Path to data folder
DATA_PATH = PROJECT_ROOT / "Data"

# Load dataset
df = pd.read_csv(DATA_PATH /"ObesityDataSet_raw_and_data_sinthetic.csv")

df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [4]:

# Binary target
df["obesity_risk"] = df["NObeyesdad"].apply(
    lambda x: 0 if x in ["Insufficient_Weight", "Normal_Weight"] else 1
)

y = df["obesity_risk"]

# Remove target original e variáveis com leakage
X = df.drop(
    columns=[
        "NObeyesdad",
        "obesity_risk",
        "Height",
        "Weight"
    ]
)

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

In [5]:
# Identify categorical columns
categorical_columns = X.select_dtypes(include="object").columns
categorical_columns

Index(['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE',
       'SCC', 'CALC', 'MTRANS'],
      dtype='object')

In [6]:
# One-Hot Encoding
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

In [7]:
X_encoded.shape

(2111, 21)

In [8]:
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

X.shape, y.shape

((2111, 14), (2111,))

#### Cross‑Validation Setup

In [9]:
k = 5

cv = StratifiedKFold(
    n_splits=k,
    shuffle=True,
    random_state=42
)

#### AutoSklearn2 — Model Selection via CV

In [10]:
cv_roc_auc = []
cv_f1 = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y), start=1):

    X_train_cv = X_encoded.iloc[train_idx]
    X_val_cv   = X_encoded.iloc[val_idx]
    y_train_cv = y.iloc[train_idx]
    y_val_cv   = y.iloc[val_idx]

    automl = AutoSklearnClassifier(
        time_limit=120,
        random_state=42
    )

    automl.fit(X_train_cv, y_train_cv)

    y_val_pred = automl.predict(X_val_cv)
    y_val_proba = automl.predict_proba(X_val_cv)[:, 1]

    cv_roc_auc.append(roc_auc_score(y_val_cv, y_val_proba))
    cv_f1.append(f1_score(y_val_cv, y_val_pred))

Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packa

#### Cross‑Validation Summary

In [11]:
print("\n=== Cross-Validation Summary ===")
print(f"Mean ROC-AUC: {np.mean(cv_roc_auc):.4f}")
print(f"ROC-AUC Std: {np.std(cv_roc_auc):.4f}")
print(f"Mean F1: {np.mean(cv_f1):.4f}")


=== Cross-Validation Summary ===
Mean ROC-AUC: 0.9779
ROC-AUC Std: 0.0059
Mean F1: 0.9509


#### Train Final AutoSklearn2 Model 

In [14]:
automl_final = AutoSklearnClassifier(
    time_limit=120,
    random_state=42
)

automl_final.fit(X_encoded, y)

Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packa

,time_limit,120
,n_jobs,-1
,random_state,42


In [20]:
print("\n=== Cross-Validation Summary ===")
print(f"Mean ROC-AUC: {np.mean(cv_roc_auc):.4f}")
print(f"ROC-AUC Std: {np.std(cv_roc_auc):.4f}")
print(f"Mean F1: {np.mean(cv_f1):.4f}")


=== Cross-Validation Summary ===
Mean ROC-AUC: 0.9779
ROC-AUC Std: 0.0059
Mean F1: 0.9509


In [21]:

print("\n=== Selected Model (AutoSklearn2) ===")
print(automl_final.best_params)



=== Selected Model (AutoSklearn2) ===
{'preprocessor': 'standard_scaler', 'classifier': 'random_forest'}


#### Model Selection Statement

Model Selection Decision

Using AutoSklearn2 with cross‑validation, the framework selected a Random Forest classifier as the best-performing model for this task.

AutoSklearn2 identified Random Forest as the best-performing model.
In the next step, I explored both Random Forest and Logistic Regression. While Random Forest offered higher predictive power, Logistic Regression provided better interpretability, which is important for risk-screening contexts.